In [44]:
!pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable
  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ------------------------------- -------- 5.2/6.6 MB 31.9 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 17.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 37.8 MB/s eta 0:00:00
Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl (183 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 25.3 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [pypdfium2]
   ---------------------------------------- 0/5 [pypdfium2]
   ---------------------------------------- 0/5 [pypdfium2]
   ---------------------------------------- 0/5 [pypdfium2]
  Attempting uninstall: cffi
   -----


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\lalan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [62]:
import os
from pathlib import Path
from uuid import uuid4
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_core.documents import Document

<h1 style="color:yellow">1. Set up Folder Path for ChromaDB </h1>

In [63]:
project_root = Path.cwd()
print(f"Project root: {project_root}")

pdf_path = project_root / "Numpy HandBook.pdf"
print(f"PDF path: {pdf_path}")

collection_name = "mmr_collection"
persist_directory = project_root / "mmr_chroma_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")



Project root: d:\LangChain Fundamentals\Retrievers in LangChain v1
PDF path: d:\LangChain Fundamentals\Retrievers in LangChain v1\Numpy HandBook.pdf
Collection name: mmr_collection
Persist directory: d:\LangChain Fundamentals\Retrievers in LangChain v1\mmr_chroma_db


<h1 style="color:yellow">2. Load PDF and Split the Text in each Document</h1>

In [71]:
loader = PDFPlumberLoader(str(pdf_path))
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=5)
documents = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_Store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),   
)

In [75]:
final_documents = []
for i, docs in enumerate(documents):
    print(f"Document {i}: {docs}")
    final_documents.append(Document(
        id=str(uuid4()),
        page_content=docs.page_content,
        metadata={"page": docs.metadata["page"], "chunk": i},
    ))

Document 0: page_content='Why Use NumPy?
Python lists are flexible but slow for numerical computing because they:
• Store elements as pointers instead of a continuous block of memory.
• Lack vectorized operations, relying on loops instead.
• Have significant overhead due to dynamic typing.
NumPy’s Superpowers:
• Faster than Python lists (C-optimized backend)
• Uses less memory (efficient storage)
• Supports vectorized operations (no explicit loops needed)
• Has built-in mathematical functions' metadata={'source': 'd:\\LangChain Fundamentals\\Retrievers in LangChain v1\\Numpy HandBook.pdf', 'file_path': 'd:\\LangChain Fundamentals\\Retrievers in LangChain v1\\Numpy HandBook.pdf', 'page': 0, 'total_pages': 23, 'Producer': 'WeasyPrint 65.0', 'Title': ' Numpy Handbook - Data Science Course'}
Document 1: page_content='NumPy vs. Python Lists – Performance Test
Let’s compare Python lists with NumPy arrays using a simple example.
Example 1: Adding Two Lists vs. NumPy Arrays
import numpy as np


<h2 style="color:yellow">View Final Documents </h2>

In [76]:
for doc in final_documents:
    print(f"Document ID: {doc.id}")
    print(f"Document Metadata: {doc.metadata}")
    print(f"Document Content: {doc.page_content[:100]}...")  # Print the first 100 characters
    print("-" * 50)

Document ID: 10133fbe-7f2f-4573-99cb-c58722e46ab8
Document Metadata: {'page': 0, 'chunk': 0}
Document Content: Why Use NumPy?
Python lists are flexible but slow for numerical computing because they:
• Store elem...
--------------------------------------------------
Document ID: 0cce7f07-106f-4fa7-9446-6f8f4af7feb9
Document Metadata: {'page': 0, 'chunk': 1}
Document Content: NumPy vs. Python Lists – Performance Test
Let’s compare Python lists with NumPy arrays using a simpl...
--------------------------------------------------
Document ID: da226119-2d21-43ee-ac3b-b0aab2a11b0c
Document Metadata: {'page': 1, 'chunk': 2}
Document Content: print("Python list addition time:", end - start)
# NumPy array
arr1 = np.array(list1)
arr2 = np.arra...
--------------------------------------------------
Document ID: a392548e-3a37-4a44-8f01-09228a52d115
Document Metadata: {'page': 1, 'chunk': 3}
Document Content: arr2 = np.array([[1, 2, 3], [4, 5, 6]])
print(arr2)
# Checking type and shape
print("Type:"

<h2 style="color:yellow">Add the Document in Chroma</h2>

In [81]:
vector_Store.add_documents(final_documents)
print(f"Total Documents added to ChromaDB successfully: {len(final_documents)}")


Total Documents added to ChromaDB successfully: 56


<h2 style="color:yellow">Applying MMR Retriever</h2>

In [92]:
query = "Dot product of two vectors ?"

retriever = vector_Store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 30, "lambda_mult": 0.5})
results = retriever.invoke(query)

print("MMR Search Results:")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Metadata: {result.metadata}")
    print(f"Content: {result.page_content}")  # Print the first 200 characters
    print("-" * 50)




MMR Search Results:
Result 1:
Metadata: {'page': 17, 'chunk': 43}
Content: Example: Broadcasting with Two Arrays
arr1 = np.array([1, 2, 3])
arr2 = np.array([10, 20, 30])
result = arr1 + arr2 # Element-wise addition
print(result) # Output: [11 22 33]
NumPy automatically aligns the two arrays and performs element-wise addition,
treating them as if they have the same shape.
Example: Broadcasting a 2D Array and a 1D Array
arr1 = np.array([[1, 2, 3], [4, 5, 6]])
arr2 = np.array([1, 2, 3])
result = arr1 + arr2 # Broadcasting arr2 across arr1
print(result)
# Output:
--------------------------------------------------
Result 2:
Metadata: {'chunk': 55, 'page': 22}
Content: These methods are used for performing mathematical and statistical operations
with NumPy
--------------------------------------------------
Result 3:
Metadata: {'page': 19, 'chunk': 50}
Content: 2. np.std() – Compute the standard deviation of an array.
np.std(arr)
3. np.var() – Compute the variance of an array.
---------------